# Correlation Creates Context

Notebook 00 showed that common-mode noise can cancel.

Notebook 07 showed that cancellation depends on noise structure rather than noise magnitude.

This notebook asks:

**How does recovery change as measurements become more or less correlated?**


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

ROOT = Path("..")
FIGURES = ROOT / "figures"
DATA = ROOT / "data"

FIGURES.mkdir(exist_ok=True)
DATA.mkdir(exist_ok=True)

rng = np.random.default_rng(42)


## Differential sensing model

We generate partially shared noise controlled by a correlation coefficient:

- ρ = 1.0 → perfectly shared
- ρ = 0.0 → independent

Differential recovery should improve as correlation increases.


In [ ]:
n = 1000
t = np.linspace(0, 10, n)

signal = 0.3 * np.sin(2 * np.pi * 0.5 * t)

rho_values = np.linspace(0, 1, 25)

rmse_values = []


In [ ]:
for rho in rho_values:

    shared = rng.normal(size=n)

    noise_a = shared

    independent = rng.normal(size=n)

    noise_b = (
        rho * shared +
        np.sqrt(1 - rho**2) * independent
    )

    local_a = 0.15 * rng.normal(size=n)
    local_b = 0.15 * rng.normal(size=n)

    phi_a = signal/2 + noise_a + local_a
    phi_b = -signal/2 + noise_b + local_b

    phi_diff = phi_a - phi_b

    rmse = np.sqrt(
        np.mean((phi_diff - signal) ** 2)
    )

    rmse_values.append(rmse)


In [ ]:
plt.figure(figsize=(7,4))

plt.plot(rho_values, rmse_values, marker="o")

plt.xlabel("Correlation coefficient (ρ)")
plt.ylabel("Differential RMSE")

plt.title("Recovery improves with correlation")

plt.tight_layout()

plt.savefig(
    FIGURES / "13_correlation_sweep.png",
    dpi=200
)

plt.show()


## Example correlation levels

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10,6))

rho_examples = [0.0, 0.3, 0.7, 1.0]

for ax, rho in zip(axes.ravel(), rho_examples):

    shared = rng.normal(size=n)

    noise_a = shared

    independent = rng.normal(size=n)

    noise_b = (
        rho * shared +
        np.sqrt(1-rho**2) * independent
    )

    phi_diff = noise_a - noise_b

    ax.plot(t[:200], phi_diff[:200])

    ax.set_title(f'ρ = {rho:.1f}')

plt.tight_layout()

plt.savefig(
    FIGURES / '13_example_correlations.png',
    dpi=200
)

plt.show()


## Correlation versus local noise

In [ ]:
rho_grid = np.linspace(0,1,40)
local_grid = np.linspace(0,2,40)

heatmap = np.zeros((40,40))

for i, rho in enumerate(rho_grid):

    for j, local_amp in enumerate(local_grid):

        shared = rng.normal(size=n)

        noise_a = shared

        independent = rng.normal(size=n)

        noise_b = (
            rho * shared +
            np.sqrt(1-rho**2) * independent
        )

        local_a = local_amp * rng.normal(size=n)
        local_b = local_amp * rng.normal(size=n)

        phi_a = signal/2 + noise_a + local_a
        phi_b = -signal/2 + noise_b + local_b

        phi_diff = phi_a - phi_b

        heatmap[j,i] = np.sqrt(
            np.mean((phi_diff-signal)**2)
        )


In [ ]:
plt.figure(figsize=(8,5))

plt.imshow(
    heatmap,
    origin="lower",
    aspect="auto",
    extent=[
        rho_grid.min(),
        rho_grid.max(),
        local_grid.min(),
        local_grid.max()
    ]
)

plt.colorbar(label="RMSE")

plt.xlabel("Correlation coefficient (ρ)")
plt.ylabel("Local noise amplitude")

plt.title("Context phase diagram")

plt.tight_layout()

plt.savefig(
    FIGURES / "13_context_phase_map.png",
    dpi=200
)

plt.show()


In [ ]:
summary = pd.DataFrame({
    'finding':[
        'higher correlation',
        'lower correlation',
        'local noise'
    ],
    'effect':[
        'better recovery',
        'worse recovery',
        'limits precision'
    ]
})

summary.to_csv(
    DATA / '13_correlation_context_summary.csv',
    index=False
)

summary


# Lesson

Common-mode rejection is not binary.

Recovery improves continuously as measurements become more correlated.

Correlation creates context.

Context determines how much apparent noise can be removed.

Notebook 17 will introduce differential baselines and show how sensor separation affects shared structure.
